Summary: Gold Transformation Step

Purpose
The Gold layer aggregates, enriches, and aligns cleaned Airbnb data from the Silver layer into stakeholder-ready tables for dashboards, modeling, and strategic analysis. It combines listing attributes with calendar and review metrics, enabling segmentation, performance tracking, and time series insights. All outputs are saved as Parquet files for efficient, schema-preserving storage and fast downstream access.


What It Does:

Centralized Setup
- Uses pathlib to resolve project paths and create the data/gold directory.
- Loads cleaned Silver tables (listings, calendar, reviews) into memory.

Calendar Aggregation
- Computes listing-level metrics: total calendar days, available days, average price.
- Derives occupancy rate as 1 - availability ratio.
- Ensures each listing has a consistent calendar summary.

Review Aggregation
- Computes review count, first review date, and last review date per listing.
- Enables maturity and engagement profiling.

Feature Consolidation
- Merges listings with calendar and review aggregates.
- Drops redundant listing_id column post-merge.
- Creates a binary superhost_flag for modeling and dashboarding.

Neighbourhood Summary
- Aggregates listing features by neighbourhood_cleansed.
- Computes average price, occupancy, review count, and superhost share.
- Enables spatial analysis and dashboard filtering.

Host Performance
- Aggregates listing metrics by host_id.
- Reattaches host_name via lookup table.
- Computes host-level KPIs: listing count, average price, total reviews, superhost rate.

Time Series Trends
- Extracts year_month from calendar and review dates.
- Aggregates monthly metrics: listings live, average price, occupancy rate, review count.
- Computes new listings per month based on first calendar appearance.
- Combines all into a unified time series table for line charts and trend analysis.

Room Type Breakdown
- Aggregates listing metrics by room_type.
- Enables bar chart comparisons and slicer logic for dashboards.


Failsafes Built In
- Path resolution: Uses pathlib for OS-safe, reproducible directory setup.
- Directory creation: Ensures data/gold exists before saving outputs.
- Column selection: Loads only necessary columns for host aggregation to optimize memory.
- Redundant column cleanup: Drops duplicate listing_id post-merge.
- Consistent prints: Each block includes a confirmation with row counts or file paths.



Alignment with Scalable, Reproducible Pipeline

- Modularity - Each transformation is isolated in its own block with clear purpose
- Reproducibility - Uses fixed paths, deterministic aggregations, and saves to Parquet
- Auditability - Prints row counts, confirms saves, and preserves schema via Parquet
- Portability - pathlib ensures compatibility across OS and environments
- Scalability - Easily extendable to new aggregations or modeling tasks
- Business Relevance - Aggregates support dashboards, segmentation, and strategic insights
- Efficiency - Parquet format enables fast reads, compact storage, and schema retention

In [1]:
# ---------------------------------------------------------
# GOLD SETUP: Imports & Path Resolution
# Purpose: Centralize setup for Gold layer transformations.
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
silver_dir = project_root / "data" / "silver"
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

In [2]:
# ---------------------------------------------------------
# GOLD INGESTION: Load Silver Tables
# Purpose: Load cleaned Silver data for aggregation.
# ---------------------------------------------------------

df_listings = pd.read_parquet(silver_dir / "listings_clean.parquet")
df_calendar = pd.read_parquet(silver_dir / "calendar_clean.parquet")
df_reviews = pd.read_parquet(silver_dir / "reviews_clean.parquet")

print("✅ Silver tables loaded")

✅ Silver tables loaded


In [3]:
# ---------------------------------------------------------
# GOLD STEP: Aggregate Calendar Metrics
# Purpose: Calculate occupancy rate and average price per listing.
# ---------------------------------------------------------

calendar_agg = (
    df_calendar
    .groupby("listing_id")
    .agg(
        calendar_days=("date", "count"),
        available_days=("available", "sum"),
        avg_price=("price", "mean")
    )
    .assign(
        occupancy_rate=lambda df: 1 - (df["available_days"] / df["calendar_days"])
    )
    .reset_index()
)

print(f"✅ Calendar metrics aggregated for {len(calendar_agg):,} listings")

✅ Calendar metrics aggregated for 94,554 listings


In [4]:
# ---------------------------------------------------------
# GOLD STEP: Aggregate Review Metrics
# Purpose: Calculate review count and first/last review dates per listing.
# ---------------------------------------------------------

review_agg = (
    df_reviews
    .groupby("listing_id")
    .agg(
        review_count=("id", "count"),
        first_review_date=("date", "min"),
        last_review_date=("date", "max")
    )
    .reset_index()
)

print(f"✅ Review metrics aggregated for {len(review_agg):,} listings")

✅ Review metrics aggregated for 70,316 listings


In [5]:
# ---------------------------------------------------------
# GOLD STEP: Combine Listing, Calendar, and Review Features
# Purpose: Create unified listing_features table.
# ---------------------------------------------------------

listing_features = (
    df_listings
    .merge(calendar_agg, how="left", left_on="id", right_on="listing_id")
    .merge(review_agg, how="left", on="listing_id")
    .drop(columns=["listing_id"])
)

listing_features["superhost_flag"] = listing_features["host_is_superhost"].eq("t").astype(int)

print(f"✅ Final listing_features table: {len(listing_features):,} rows")

✅ Final listing_features table: 94,559 rows


In [6]:
# ---------------------------------------------------------
# GOLD STEP: Save listing_features to Disk
# Purpose: Store final Gold table for dashboards, ML, or portfolio use.
# ---------------------------------------------------------

output_path = gold_dir / "listing_features.parquet"
listing_features.to_parquet(output_path, index=False)

print(f"✅ Saved listing_features to {output_path}")

✅ Saved listing_features to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\listing_features.parquet


In [7]:
# ---------------------------------------------------------
# GOLD STEP: Neighbourhood Summary
# Purpose: Aggregate listing features by neighbourhood.
# ---------------------------------------------------------

df_lf = listing_features

neighbourhood_summary = (
    df_lf
    .groupby("neighbourhood_cleansed")
    .agg(
        total_listings        = ("id",            "nunique"),
        avg_price             = ("avg_price",     "mean"),
        median_price          = ("avg_price",     "median"),
        avg_occupancy_rate    = ("occupancy_rate","mean"),
        avg_review_count      = ("review_count",  "mean"),
        superhost_share       = ("superhost_flag","mean")
    )
    .reset_index()
)

print(f"✅ Neighbourhood summary created: {len(neighbourhood_summary):,} rows")

output_path = gold_dir / "neighbourhood_summary.parquet"
neighbourhood_summary.to_parquet(output_path, index=False)

print(f"✅ Saved neighbourhood_summary to {output_path}")

✅ Neighbourhood summary created: 33 rows
✅ Saved neighbourhood_summary to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\neighbourhood_summary.parquet


C:\Users\emand\AppData\Local\Temp\ipykernel_22828\872219704.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("neighbourhood_cleansed")


In [8]:
# ---------------------------------------------------------
# GOLD STEP: Host Performance
# Purpose: Aggregate listing metrics by host_id.
# ---------------------------------------------------------

cols = [
    "id", "host_id", "host_name",
    "avg_price", "occupancy_rate",
    "review_count", "superhost_flag"
]
df = pd.read_parquet(gold_dir / "listing_features.parquet", columns=cols)

host_lookup = (
    df[["host_id", "host_name"]]
    .drop_duplicates(subset="host_id")
    .set_index("host_id")
)

host_perf = (
    df
    .groupby("host_id", sort=False)
    .agg(
        listing_count        = ("id",              "nunique"),
        avg_price            = ("avg_price",       "mean"),
        avg_occupancy_rate   = ("occupancy_rate",  "mean"),
        total_review_count   = ("review_count",    "sum"),
        superhost_rate       = ("superhost_flag",  "mean")
    )
    .join(host_lookup, how="left")
    .reset_index()
)

output_path = gold_dir / "host_performance.parquet"
host_perf.to_parquet(output_path, index=False)

print(f"✅ {len(host_perf):,} hosts saved to {output_path.name}")

✅ 55,395 hosts saved to host_performance.parquet


In [9]:
# ---------------------------------------------------------
# GOLD STEP: Time Series Trends
# Purpose: Compute monthly market dynamics for line charts.
# ---------------------------------------------------------

df_calendar["year_month"] = df_calendar["date"].dt.to_period("M").astype(str)
df_reviews["year_month"] = df_reviews["date"].dt.to_period("M").astype(str)

calendar_ts = (
    df_calendar
    .groupby("year_month")
    .agg(
        listings_live=("listing_id", "nunique"),
        avg_price=("price", "mean"),
        avg_occupancy_rate=("available", lambda x: 1 - x.sum() / x.count())
    )
    .reset_index()
)

reviews_ts = (
    df_reviews
    .groupby("year_month")
    .agg(review_count=("id", "count"))
    .reset_index()
)

first_calendar = (
    df_calendar
    .groupby("listing_id")["date"]
    .min()
    .dt.to_period("M")
    .astype(str)
    .reset_index(name="first_month")
)

new_listings = (
    first_calendar
    .groupby("first_month")
    .size()
    .reset_index(name="new_listings")
    .rename(columns={"first_month": "year_month"})
)

time_series = (
    calendar_ts
    .merge(reviews_ts, on="year_month", how="left")
    .merge(new_listings, on="year_month", how="left")
)

print(f"✅ Time series table created: {len(time_series):,} months")

output_path = gold_dir / "time_series.parquet"
time_series.to_parquet(output_path, index=False)

print(f"✅ Saved time_series to {output_path}")

✅ Time series table created: 13 months
✅ Saved time_series to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\time_series.parquet


In [10]:
# ---------------------------------------------------------
# GOLD STEP: Room Type Breakdown
# Purpose: Compare performance across room types.
# ---------------------------------------------------------

room_type_breakdown = (
    df_lf
    .groupby("room_type")
    .agg(
        listing_count=("id", "nunique"),
        avg_price=("avg_price", "mean"),
        avg_occupancy_rate=("occupancy_rate", "mean"),
        avg_review_count=("review_count", "mean")
    )
    .reset_index()
)

print(f"✅ Room type breakdown created: {len(room_type_breakdown):,} categories")

output_path = gold_dir / "room_type_breakdown.parquet"
room_type_breakdown.to_parquet(output_path, index=False)

print(f"✅ Saved room_type_breakdown to {output_path}")

✅ Room type breakdown created: 4 categories
✅ Saved room_type_breakdown to C:\Users\emand\Documents\Python\DSPP  - AirBnBProject\data\gold\room_type_breakdown.parquet


C:\Users\emand\AppData\Local\Temp\ipykernel_22828\4196108895.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("room_type")
